# Reprodukowalność w Data Science

#### Reprodukowalność wyników jest kluczowa w projektach Data Science. 

#### Pozwala ona na odtworzenie wyników przez innych (lub przez Ciebie w przyszłości) przy użyciu tych samych danych, kodu i środowiska. 


In [2]:
import pandas as pd
import random
import numpy as np
from faker import Faker
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from pycaret.classification import setup
from pycaret.datasets import get_data

In [3]:
wine_df = get_data('wine', verbose=False)
wine_df.sample(5, random_state=42)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
3103,7.0,0.17,0.74,12.8,0.045,24.0,126.0,0.99420,3.26,0.38,12.2,8,white
1419,7.7,0.64,0.21,2.2,0.077,32.0,133.0,0.99560,3.27,0.45,9.9,5,red
4761,6.8,0.39,0.34,7.4,0.020,38.0,133.0,0.99212,3.18,0.44,12.0,7,white
4690,6.3,0.28,0.47,11.2,0.040,61.0,183.0,0.99592,3.12,0.51,9.5,6,white
4032,7.4,0.35,0.20,13.9,0.054,63.0,229.0,0.99888,3.11,0.50,8.9,6,white


## Dane i wersjonowanie

#### Zadbaj o stałą wersję danych

Zawsze upewnij się, że korzystasz z tej samej wersji danych na każdym etapie projektu:
- Przechowuj dane w dedykowanym folderze w projekcie.
- Jeśli dane mogą się zmieniać (np. zewnętrzne pliki CSV), zapisz ich datę pobrania lub wersję.
- Jeśli dokonujesz przetwarzania danych (np. oczyszczanie), zapisuj kod transformacji, aby zawsze móc wykonać te same operacje w przyszłości.
- Załóżmy, że zbierasz różne wersje danych wówczas możemy wprowadzić następujący format
```
sales/
    2021-01-01/
        data.parquet
        metadata.json
    2021-02-01/
        data.parquet
        metadata.json
```

Gdzie `data.parquet` to plik z danymi, a `metadata.json` to plik z metadanymi, takimi jak liczba wierszy, kolumn, itp.

### Rozważmy przykład wygenerowania `metadata.json`

In [ ]:
happy_df = pd.read_csv('world_happiness_report.csv', sep=";")

meta_data = {
    "rows": happy_df.shape[0],
    "columns": happy_df.shape[1],
    "min_year": happy_df['Year'].min(),
    "max_year": happy_df['Year'].max(),
    "countries": happy_df['Country name'].nunique(),
}
meta_data

## Kontrolowanie losowości

#### Innym kluczowym aspektem zapewnienia reprodukowalności jest ustawienie **seed** dla operacji losowych.

#### Innymi słowy, seed to liczba, która inicjalizuje generator liczb losowych, co pozwala na uzyskanie tych samych wyników przy kolejnych uruchomieniach.

#### Czyli każdy kto uruchomi nasz kod z takim samym seedem, otrzyma te same wyniki!

#### Reprodukowalności w module `random`

In [ ]:
random.seed(42)

# OCZEKUJEMY: 81
random.randint(0, 100)

#### Reprodukowalności w bibliotece `numpy`

In [ ]:
np.random.seed(42)

# OCZEKUJEMY: array([0.37454012, 0.95071431, 0.73199394, 0.59865848, 0.15601864])
np.random.rand(5)

#### Reprodukowalności w bibliotece `faker`

In [ ]:
fake = Faker()
Faker.seed(42)

# OCZEKUJEMY: 'Allison Hill'
fake.name()

#### Reprodukowalności w bibliotece `scikit-learn`

In [ ]:
X, y = wine_df.drop(columns=['type']), wine_df['type']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# OCZEKUJEMY: 0.9946153846153846
model.score(X_test, y_test)

#### Reprodukowalności w bibliotece `pycaret`

In [ ]:
exp = setup(data=wine_df, target='type', session_id=42, verbose=False)

# OCZEKUJEMY: 0.9945, 0.0000, 0.9945, 0.9945, 0.9945, 0.9851, 0.9852
exp.create_model("rf")

## Środowisko

#### Po aktywacji środowiska, zapisz jego konfigurację do pliku `environment.yaml`:

```bash
conda activate od_zera_do_ai
conda env export > environment.yaml
```

#### Następnie na innym komputerze, aby odtworzyć środowisko, wykonaj:

```bash
conda env create -f environment.yaml
```

#### Zobaczmy jak wygląda plik `environment.yml`

In [ ]:
with open('environment.yaml', 'r') as file:
    print(file.read()[:1000])